In [ ]:
# 1. Установка (самая свежая версия Unsloth с поддержкой Qwen3)
pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
# Для Windows/Linux с CUDA 12.1+
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# 2. Код инференса (работает одинаково для всех размеров)
from unsloth import FastLanguageModel
import torch

# Выбери нужную (пример для 32B — самая популярная на данный момент)
model_name = "unsloth/Qwen3-32B-Instruct-bnb-4bit"   # или 8B, 14B, 72B

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=32768,      # Qwen3 поддерживает до 128k, но 32k достаточно и быстрее
    dtype=None,                # авто: bf16 на новых картах, fp16 на старых
    load_in_4bit=True,         # уже 4-bit, не меняй
    device_map="auto",
    # token=hf_token,         # если модель gated (пока Qwen3 открытая)
)

# Включаем максимальное ускорение инференса (до 3.2× быстрее + меньше VRAM)
FastLanguageModel.for_inference(model)

# Qwen3 использует новый чат-шаблон (обязательно apply_chat_template!)
messages = [
    {"role": "system", "content": "Ты Grok, но на базе Qwen3. Отвечай максимально полезно и честно."},
    {"role": "user", "content": "Расскажи про ключевые улучшения Qwen3 по сравнению с Qwen2.5"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,   # без этого модель не начнёт генерировать ответ
    return_tensors="pt"
).to(model.device)

# Стриминг + быстрые параметры (рекомендуемые для Qwen3)
from transformers import TextStreamer
streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

_ = model.generate(
    inputs,
    max_new_tokens=2048,
    streamer=streamer,
    temperature=0.7,
    top_p=0.9,
    top_k=40,
    repetition_penalty=1.05,
    do_sample=True,
    use_cache=True,
)

# хз че

In [ ]:
!pip install gradio -q

import gradio as gr

def chat_with_qwen3(message, history):
    conversation = [{"role": "system", "content": "Ты мощный ИИ Qwen3."}]
    for user, assistant in history:
        conversation.extend([{"role": "user", "content": user}, {"role": "assistant", "content": assistant}])
    conversation.append({"role": "user", "content": message})

    input_ids = tokenizer.apply_chat_template(
        conversation,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    
    _ = model.generate(
        input_ids,
        max_new_tokens=2048,
        streamer=streamer,
        temperature=0.7,
        top_p=0.9,
        top_k=40,
        repetition_penalty=1.05,
    )

gr.ChatInterface(chat_with_qwen3, title="Qwen3-32B (Unsloth 4-bit)").launch()